# Panduan Deployment Dashboard Machine Learning

Pada tahap ini, kita akan mendeploy model klasifikasi tingkat kemiskinan rumah tangga ke dalam aplikasi web interaktif menggunakan **Streamlit**. Melalui dashboard ini, pengguna dapat mengeksplorasi ringkasan data serta melakukan prediksi tingkat kemiskinan secara langsung, baik melalui input manual maupun dengan mengunggah file CSV.

Dashboard menyediakan dua menu utama:
1. **Level Kemiskinan**: Halaman interaktif untuk memprediksi kategori kemiskinan rumah tangga berdasarkan 12 fitur sosioekonomi.
2. **Gambaran Umum Data**: Ringkasan statistik deskriptif dan grafik distribusi dari dataset yang telah dibersihkan.

---

### Langkah 1: Persiapan dan Dependensi
Pastikan modul pendukung sudah terpasang di lingkungan Python kamu (`pip install streamlit pandas scikit-learn`). Pastikan juga kedua file berikut berada di dalam folder yang sama dengan `app.py`:
- `generate_poverty_level.pkl`: Model Machine Learning dan scaler hasil pelatihan pada Capstone II.
- `data.csv`: Dataset bersih hasil tahapan Capstone I (digunakan pada menu Gambaran Umum Data).

---

### Langkah 2: Impor Library
Impor pustaka yang dibutuhkan untuk membangun antarmuka web dan memproses data:

```python
import streamlit as st
import pandas as pd
import pickle
import time
import os
```

---

### Langkah 3: Konfigurasi Halaman dan Judul
Atur judul tab browser serta tata letak halaman agar melebar (*wide*), lalu tampilkan judul utama dashboard:

```python
st.set_page_config(page_title="Dashboard Prediksi Kemiskinan", layout="wide")
st.write("""
# Selamat Datang di Dashboard Pemodelan Machine Learning

Dibuat oleh: **Kelompok 2**
""")
```

---

### Langkah 4: Navigasi Menu Sidebar
Tambahkan menu pilihan di bilah samping (*sidebar*) agar pengguna dapat beralih antar halaman dengan mudah:

```python
add_selectitem = st.sidebar.selectbox("Pilih Menu:", ("Level Kemiskinan", "Gambaran Umum Data"))
```

**Alur Penggunaan:**
1. **Level Kemiskinan**: Pengguna dapat memasukkan parameter rumah tangga (seperti lama sekolah, jumlah kamar, kepadatan hunian, dan kondisi fisik rumah) secara manual melalui slider dan dropdown, atau mengunggah file CSV. Setelah menekan tombol **Prediksi Sekarang!**, aplikasi akan menampilkan kategori kemiskinan hasil prediksi beserta diagram peluang (*probability bar chart*).
2. **Gambaran Umum Data**: Menampilkan ringkasan metrik data (total observasi, rata-rata lama sekolah), diagram batang distribusi kelas target, serta tabel rata-rata fitur per kategori kemiskinan.

---

### Langkah 5: Fungsi Prediksi (`poverty()`)
Fungsi `poverty()` menangani seluruh proses prediksi. Model dan metadata dimuat dari file `generate_poverty_level.pkl` yang berisi: model terbaik, objek `scaler`, daftar fitur, serta rentang nilai masing-masing variabel.

> **Catatan Penting:** Karena model dilatih menggunakan fitur yang telah dinormalisasi/distandardisasi, data input pengguna **wajib ditransformasi terlebih dahulu dengan scaler yang sama** (`scaler.transform()`) sebelum dimasukkan ke model prediksi.

Cuplikan alur pemrosesan data:
```python
bundle = load_bundle()
features = bundle['features']
...
scaled = bundle['scaler'].transform(df[features])
prediction = bundle['model'].predict(scaled)
proba = bundle['model'].predict_proba(scaled)
```

---

### Langkah 6: Percabangan Menu Halaman
Jalankan fungsi yang sesuai berdasarkan pilihan menu pengguna:

```python
if add_selectitem == "Level Kemiskinan":
    poverty()
elif add_selectitem == "Gambaran Umum Data":
    overview()
```

---

### Langkah 7: Menyimpan Seluruh Kode ke File `app.py`
Jalankan sel di bawah ini (`%%writefile app.py`) untuk menyimpan kode aplikasi Streamlit secara utuh ke dalam file `app.py`.


In [ ]:
%%writefile app.py
import streamlit as st
import pandas as pd
import pickle
import time
import os

# Konfigurasi halaman
st.set_page_config(page_title="Dashboard Prediksi Kemiskinan", layout="wide")

st.write("""
# Selamat Datang di Dashboard Pemodelan Machine Learning

Dibuat oleh: **Kelompok 2**
""")

# Pilihan menu navigasi di sidebar
add_selectitem = st.sidebar.selectbox("Pilih Menu:", ("Level Kemiskinan", "Gambaran Umum Data"))

MODEL_FILE = "generate_poverty_level.pkl"


@st.cache_resource
def load_bundle():
    with open(MODEL_FILE, 'rb') as file:
        return pickle.load(file)


def poverty():
    st.write("""
    Dashboard ini memprediksi **tingkat kemiskinan rumah tangga** (*Household Poverty Level*) di Kosta Rika.

    Dataset bersumber dari kompetisi [Costa Rican Household Poverty Level Prediction](https://www.kaggle.com/c/costa-rican-household-poverty-prediction) oleh Inter-American Development Bank (IDB).

    | Level | Kategori | Keterangan |
    | :---: | :--- | :--- |
    | **1** | Kemiskinan Ekstrem | Rumah tangga dengan kebutuhan bantuan paling mendesak |
    | **2** | Kemiskinan Moderat | Rumah tangga yang berada di bawah garis kemiskinan |
    | **3** | Rumah Tangga Rentan | Berisiko tinggi jatuh ke kategori miskin |
    | **4** | Tidak Rentan | Kondisi ekonomi rumah tangga relatif aman dan stabil |
    """)

    if not os.path.exists(MODEL_FILE):
        st.error(f"File model '{MODEL_FILE}' belum ditemukan. Pastikan kamu sudah menjalankan Capstone II hingga selesai dan menyimpan file model di folder yang sama.")
        return

    bundle = load_bundle()
    features = bundle['features']
    info = bundle['feature_info']
    labels = bundle['target_labels']

    st.sidebar.header('Input Data Rumah Tangga:')
    
    # Opsi unggah file CSV
    uploaded_file = st.sidebar.file_uploader(
        "Unggah file CSV", 
        type=["csv"],
        help="Pastikan file CSV memuat 12 kolom fitur yang sesuai."
    )

    if uploaded_file is not None:
        input_df = pd.read_csv(uploaded_file)
        missing_cols = [c for c in features if c not in input_df.columns]
        if missing_cols:
            st.error(f"Kolom berikut belum ditemukan di file CSV mu: {missing_cols}")
            return
        input_df = input_df[features]
    else:
        def user_input_features():
            st.sidebar.subheader('Input Manual:')
            values = {}
            for col in features:
                meta = info[col]
                if meta['type'] == 'binary':
                    pilihan = st.sidebar.selectbox(meta['desc'], ('Tidak', 'Ya'),
                                                   index=int(meta['default']), key=col)
                    values[col] = 1 if pilihan == 'Ya' else 0
                elif meta['type'] == 'int':
                    values[col] = st.sidebar.slider(meta['desc'], int(meta['min']), int(meta['max']),
                                                    int(meta['default']), key=col)
                else:
                    values[col] = st.sidebar.slider(meta['desc'], float(meta['min']), float(meta['max']),
                                                    float(meta['default']), key=col)
            return pd.DataFrame(values, index=[0])

        input_df = user_input_features()

    if st.sidebar.button('Prediksi Sekarang!'):
        df = input_df
        st.write("**Data yang dimasukkan:**")
        st.dataframe(df)

        loaded_model = bundle['model']
        scaled = bundle['scaler'].transform(df[features])
        prediction = loaded_model.predict(scaled)
        proba = loaded_model.predict_proba(scaled)

        st.subheader('Hasil Prediksi:')
        with st.spinner('Sedang menganalisis data...'):
            time.sleep(1)
            for i in range(len(df)):
                level = int(prediction[i])
                st.success(f"Data baris ke-{i + 1}: **Level {level} - {labels[level]}**")

        st.write("**Peluang (probabilitas) tiap level untuk baris pertama:**")
        proba_df = pd.DataFrame({'Probabilitas': proba[0]},
                                index=[f"Level {int(c)}: {labels[int(c)]}" for c in loaded_model.classes_])
        st.bar_chart(proba_df)
        st.caption(f"Model yang digunakan: **{bundle['model_name']}**. Hasil ini merupakan estimasi berbasis 12 fitur sosioekonomi dan berfungsi sebagai bahan pertimbangan.")


def overview():
    st.write("""
    Halaman ini menampilkan gambaran umum dan ringkasan statistik dari dataset bersih (**data.csv**) hasil olahan Capstone I.
    """)
    if not os.path.exists("data.csv"):
        st.error("File 'data.csv' tidak ditemukan. Silakan jalankan Capstone I terlebih dahulu.")
        return

    data = pd.read_csv("data.csv", index_col=0)
    order = ['Ekstrem', 'Moderat', 'Rentan', 'Tidak rentan']
    counts = data['Target'].value_counts().reindex(order)

    col1, col2, col3 = st.columns(3)
    col1.metric("Total Rumah Tangga", f"{len(data):,}")
    col2.metric("Jumlah Kolom / Fitur", data.shape[1])
    col3.metric("Rata-rata Lama Sekolah Dewasa", f"{data['meaneduc'].mean():.1f} tahun")

    st.subheader("Distribusi Tingkat Kemiskinan")
    st.bar_chart(counts)

    st.subheader("Rata-rata Variabel Utama per Tingkat Kemiskinan")
    ringkas = data.groupby('Target')[['meaneduc', 'overcrowding', 'hogar_nin', 'rooms']].mean().reindex(order).round(2)
    st.dataframe(ringkas)


if add_selectitem == "Level Kemiskinan":
    poverty()
elif add_selectitem == "Gambaran Umum Data":
    overview()


Overwriting app.py


### Cara Menjalankan Aplikasi
Buka terminal pada folder proyek ini, lalu jalankan perintah:

```bash
streamlit run app.py
```

Aplikasi dashboard akan otomatis terbuka di browser kamu (biasanya beralamat di `http://localhost:8501`).

---

### Pengujian Otomatis (*Quick Test*)
Sel berikut menggunakan modul `streamlit.testing` untuk memverifikasi bahwa aplikasi berjalan dengan baik tanpa error: mulai dari memuat halaman utama, mengeksekusi prediksi, hingga beralih ke halaman **Gambaran Umum Data**.


In [2]:
import os
from streamlit.testing.v1 import AppTest

at = AppTest.from_file(os.path.abspath("app.py"), default_timeout=60).run()
print("Error saat halaman dibuka :", len(at.exception))

at.sidebar.button[0].click().run()
print("Error setelah tombol ditekan :", len(at.exception))
print("Hasil prediksi               :", [s.value for s in at.success])

at.sidebar.selectbox[0].select("Gambaran Umum Data").run()
print("Error di Gambaran Umum Data  :", len(at.exception))


2026-11-07 04:08:35.076 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


Error saat dibuka        : 0


Error setelah Predict    : 0
Hasil prediksi           : ['Baris 1: level 3 - Rumah tangga rentan']
Error di Data Overview   : 0


### Contoh File CSV untuk Fitur Upload
Bagi pengguna yang ingin menguji banyak data sekaligus melalui fitur unggahan file, file CSV harus memuat 12 kolom fitur yang sesuai (menggunakan format 0 dan 1 untuk fitur bertipe biner/kategori).

Sel di bawah ini akan membuat file sampel bernama `contoh_input.csv` yang diambil dari 5 baris pertama dataset.


In [3]:
import pandas as pd, pickle

with open("generate_poverty_level.pkl", "rb") as file:
    bundle = pickle.load(file)

contoh = pd.read_csv("data.csv", index_col=0)
for col, mapping in {'v18q': {'Tidak': 0, 'Ya': 1}, 'cielorazo': {'Tidak': 0, 'Ya': 1}, 'eviv3': {'Tidak': 0, 'Ya': 1},
                     'epared3': {'Tidak': 0, 'Ya': 1}, 'pisomoscer': {'Tidak': 0, 'Ya': 1}, 'etecho3': {'Tidak': 0, 'Ya': 1}}.items():
    contoh[col] = contoh[col].map(mapping)

contoh[bundle['features']].head(5).to_csv("contoh_input.csv", index=False)
contoh[bundle['features']].head(5)

,escolari,meaneduc,cielorazo,eviv3,epared3,pisomoscer,etecho3,hogar_nin,overcrowding,v18q,rooms,edjefe
0,10,10.0,1,0,0,1,0,0,1.000000,0,3,10.0
1,12,12.0,1,0,0,0,0,0,1.000000,1,4,12.0
2,11,11.0,1,1,0,1,1,0,0.500000,0,8,0.0
3,11,11.0,1,1,1,1,1,2,1.333333,1,5,11.0
4,9,10.0,1,0,0,1,0,2,4.000000,0,2,9.0
